# Grouped-Query Attention (GQA) test

This notebook creates the GQA variant, checks that cached decoding matches full decoding, and generates a short sample. The model starts with random weights; train it or load GQA weights for meaningful text.

In [1]:
import time

import tiktoken
import torch

from pathlib import Path
import sys

optimization_root = Path.cwd().resolve().parent
if str(optimization_root) not in sys.path:
    sys.path.insert(0, str(optimization_root))

from Transformer_arquitectures import GQAGPTModel
from help_functions import generate_text_simple_cached


In [2]:
# n_kv_groups must divide n_heads exactly.
# 12 groups = regular MHA; 1 group = MQA; 2/3/4/6 = GQA.
GPT_CONFIG_GQA = {
    'vocab_size': 50257,
    'context_length': 256,
    'emb_dim': 768,
    'n_heads': 12,
    'n_kv_groups': 2,
    'n_layers': 12,
    'drop_rate': 0.0,
    'qkv_bias': False,
}

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
torch.manual_seed(42)
model = GQAGPTModel(GPT_CONFIG_GQA).to(device).eval()
print(f'Using device: {device}')
print(f'Query heads: {GPT_CONFIG_GQA["n_heads"]}')
print(f'KV groups: {GPT_CONFIG_GQA["n_kv_groups"]}')


Using device: mps
Query heads: 12
KV groups: 2


In [3]:
# Verify that autoregressive cache decoding equals a full forward pass.
test_idx = torch.randint(
    0, GPT_CONFIG_GQA['vocab_size'], (1, 8), device=device
)

with torch.no_grad():
    full_logits = model(test_idx)

    model.reset_kv_cache()
    cached_logits = torch.cat([
        model(test_idx[:, step:step + 1], use_cache=True)
        for step in range(test_idx.shape[1])
    ], dim=1)

max_difference = (full_logits - cached_logits).abs().max().item()
print(f'Maximum difference: {max_difference:.2e}')
assert torch.allclose(full_logits, cached_logits, atol=1e-5)
print('GQA KV cache check passed.')


Maximum difference: 7.15e-07
GQA KV cache check passed.


In [4]:
tokenizer = tiktoken.get_encoding('gpt2')
prompt = 'Hello, I am'
idx = torch.tensor(
    tokenizer.encode(prompt), dtype=torch.long, device=device
).unsqueeze(0)

if device.type == 'cuda':
    torch.cuda.synchronize()
start = time.perf_counter()

output_ids = generate_text_simple_cached(
    model=model,
    idx=idx,
    max_new_tokens=50,
    context_size=GPT_CONFIG_GQA['context_length'],
    use_cache=True,
)

if device.type == 'cuda':
    torch.cuda.synchronize()
elapsed = time.perf_counter() - start

print(tokenizer.decode(output_ids[0].tolist()))
print(f'Generated 50 tokens in {elapsed:.2f} seconds.')
print('The output is random until this GQA model is trained.')


Hello, I amstorm store irritSelf Perform submer Lopezheses accumulation HB CM strainsuf Perl currencyables snapshotsedition013 splitting Lavrov2001 watts Aura Sustainableothing notor assumedigrants MAP Huff dogs softenourdoiScript Zur landmarksmetic guideline mot Anders Gitpoons Uz escalatingBuildiling Kos Qaeda
Generated 50 tokens in 1.05 seconds.
The output is random until this GQA model is trained.
